In [1]:
# =============================================================================
# CELL 1 -- setup: tiny parameter space, low resolution, synthetic data
# =============================================================================
# A TEST HARNESS for nested sampling on the four faculae-band parameters
#   inc [deg], v_eq [m/s], lat_centre [deg], ff
# The prior is a deliberately NARROW box around the truth and the resolution is
# deliberately LOW, so a full run takes minutes. Neither is a physics choice:
# nothing here is a precision estimate, it only checks that the sampler machinery
# runs, recovers an injected signal, and gives a sane log Z.

import time

import numpy as np
import matplotlib.pyplot as plt

import actress as ac   # REFRESH BEFORE USE!

SEED = 20260922

# --- resolution: LOW, for speed. Raises the healpix pixelisation floor. ---
RES_3D = 5      # nside = 2**RES_3D = 32
RES_2D = 200    # projected disc image side, pixels

# --- limb darkening, single line, coefficients held fixed ---
LINE_WAVELENGTH_ANG = 4375.0
hd_ld_table  = np.loadtxt('test_phot.txt')       # cols [wavelength /m, I0, a, b]
fac_ld_table = np.loadtxt('test_fac_200G.txt')
ld_row = int(np.argmin(np.abs(hd_ld_table[:, 0] * 1.0e10 - LINE_WAVELENGTH_ANG)))
PHOT_COEFFS = list(hd_ld_table[ld_row, 1:4])
FAC_COEFFS  = list(fac_ld_table[ld_row, 1:4])

def power2_ld(mu, I0, a, b):
    return I0 * (1.0 - (a * (1.0 - (mu ** b))))

# One dict, passed to the Simulator constructor. The three limb-darkening masks
# depend only on this and on xsize, so they are built once per Simulator.
LD_DICT = {'phot': PHOT_COEFFS, 'spot': [1.0e7, 0.3, 0.2],
           'fac': FAC_COEFFS, 'func': power2_ld}

# --- line profile binning ---
N_PROFILE_BINS = 20
PROFILE_HALF_WIDTH_ANG = 0.09
PROFILE_BIN_EDGES = np.linspace(-PROFILE_HALF_WIDTH_ANG, PROFILE_HALF_WIDTH_ANG,
                                N_PROFILE_BINS + 1)

# --- the injected truth ---
TRUTH = np.array([75.0, 4000.0, 25.0, 0.12])     # inc, v_eq, lat_centre, ff
LABELS = ['inc [deg]', 'v_eq [m/s]', 'lat_centre [deg]', 'ff']
N_DIM = 4

# --- ARTIFICIALLY SMALL prior: a narrow uniform box bracketing the truth ---
# Chosen so the posterior is well inside the box and the sampler has little volume
# to compress. Widen these to make the test harder.
PRIOR_LO = np.array([70.0, 3600.0, 21.0, 0.100])
PRIOR_HI = np.array([80.0, 4400.0, 29.0, 0.140])

# --- noise ---
SIGMA_FLUX_REL    = 1.0e-4
SIGMA_PROFILE_REL = 1.0e-3


def band_edges_from_ff(lat_centre_deg, ff):
    """
    Latitude edges [deg] of the band of filling factor ff centred on lat_centre_deg.
    Half-width is arcsin(ff / cos(lat_c)) from f = cos(lat_c) sin(h) (Archimedes).

    Raises if an edge leaves +/-90: hp.query_strip does NOT raise there, it silently
    returns the band reflected about 90 deg with the same area and wrong latitudes.
    The binding condition is ff <= cos^2(lat_c), not ff <= cos(lat_c).
    """
    cos_lat = np.cos(np.radians(lat_centre_deg))
    if ff > cos_lat:
        raise ValueError('ff={:.4f} unreachable at lat={:.2f} (max cos(lat)={:.4f})'
                         .format(ff, lat_centre_deg, cos_lat))
    half_width = np.degrees(np.arcsin(ff / cos_lat))
    lat_low, lat_high = lat_centre_deg - half_width, lat_centre_deg + half_width
    if lat_low < -90.0 or lat_high > 90.0:
        raise ValueError('band {:.2f} to {:.2f} leaves +/-90 for lat={:.2f}, ff={:.4f}; '
                         'limit here is cos^2(lat)={:.4f}'
                         .format(lat_low, lat_high, lat_centre_deg, ff, cos_lat ** 2))
    return lat_low, lat_high


# the whole prior box must be geometrically valid, or the test is probing a region
# where the forward model is not what theta says it is
_lat_hi_corner = PRIOR_HI[2]
if PRIOR_HI[3] > np.cos(np.radians(_lat_hi_corner)) ** 2:
    raise ValueError('prior box corner (lat={}, ff={}) exceeds cos^2(lat)={:.4f}'
                     .format(_lat_hi_corner, PRIOR_HI[3],
                             np.cos(np.radians(_lat_hi_corner)) ** 2))


def forward_model(theta):
    """
    theta = (inc [deg], v_eq [m/s], lat_centre [deg], ff)
    Returns (flux_excess, profile), profile normalised to unit sum.
    Exceptions propagate: a model failure inside the prior is a bug, not a region
    to be quietly excluded.
    """
    inc_deg, v_eq_ms, lat_centre_deg, ff = theta
    lat_low, lat_high = band_edges_from_ff(lat_centre_deg, ff)

    # xsize and ld go to the CONSTRUCTOR. Passing them here builds the three
    # limb-darkening masks once, at RES_2D. Calling setxsize/setld afterwards instead
    # builds them first at the xsize=800 default (simulator.py:70, 117, 128-130),
    # throws that away, and rebuilds twice more: ~12x the cost per likelihood call.
    #
    # resolution CANNOT go in the constructor: simulator.py:123 validates the argument
    # and then hardcodes 2**6, so setresolution is still required or you silently get
    # nside=64 regardless of RES_3D.
    sim = ac.Simulator(faculae=[], spots=[], fac_strips=[], spot_strips=[],
                       xsize=RES_2D, ld=LD_DICT)
    sim.setresolution(RES_3D)
    sim.addstrip(lower=lat_low, upper=lat_high, feature='fac')

    discs, shift_map = sim.rotate_lc(v_eq=v_eq_ms, inc=inc_deg, N=1, mode='faconly',
                                     wavelength=LINE_WAVELENGTH_ANG,
                                     returndisc=True, njobs=1)
    disc = np.asarray(discs)[0]

    bin_index = np.digitize(shift_map.ravel(), PROFILE_BIN_EDGES) - 1
    in_window = (bin_index >= 0) & (bin_index < N_PROFILE_BINS)
    flat = disc.ravel()
    profile = np.bincount(bin_index[in_window], weights=flat[in_window],
                          minlength=N_PROFILE_BINS)

    total = np.nansum(flat)
    if total - profile.sum() > 1.0e-6 * total:
        raise ValueError('{:.3%} of disc flux falls outside the profile window'
                         .format((total - profile.sum()) / total))

    return float(np.nanmean(disc)) / QUIET_FLUX, profile / profile.sum()


# featureless star: uniform, so identical at every phase and inclination -> one scalar
_sim = ac.Simulator(faculae=[], spots=[], fac_strips=[], spot_strips=[],
                    xsize=RES_2D, ld=LD_DICT)
_sim.setresolution(RES_3D)
_quiet, _ = _sim.rotate_lc(v_eq=0.0, inc=90.0, N=1, mode='quiet',
                           wavelength=LINE_WAVELENGTH_ANG, returndisc=True, njobs=1)
QUIET_FLUX = float(np.nanmean(np.asarray(_quiet)[0]))

# --- synthetic data ---
rng = np.random.default_rng(SEED)
flux_model, profile_model = forward_model(TRUTH)
FLUX_SIGMA = SIGMA_FLUX_REL * flux_model
PROFILE_SIGMA = SIGMA_PROFILE_REL * profile_model.max()
FLUX_DATA = flux_model + rng.normal(0.0, FLUX_SIGMA)
PROFILE_DATA = profile_model + rng.normal(0.0, PROFILE_SIGMA, size=profile_model.shape)


def prior_transform(u):
    """Unit cube -> parameters. Uniform on the narrow box; no constraint needed,
    the box was checked against cos^2(lat) above."""
    return PRIOR_LO + u * (PRIOR_HI - PRIOR_LO)


def log_likelihood(theta):
    """Chi-squared over flux excess and profile shape, independent Gaussian errors."""
    flux, profile = forward_model(theta)
    chi2 = ((FLUX_DATA - flux) / FLUX_SIGMA) ** 2
    chi2 += np.sum(((PROFILE_DATA - profile) / PROFILE_SIGMA) ** 2)
    return -0.5 * chi2


_t0 = time.time()
_ = log_likelihood(TRUTH)
T_LIKE = time.time() - _t0

print('quiet flux            : {:.6f}'.format(QUIET_FLUX))
print('data vector           : 1 + {} = {} numbers, {} parameters'
      .format(N_PROFILE_BINS, 1 + N_PROFILE_BINS, N_DIM))
print('one likelihood call   : {:.4f} s'.format(T_LIKE))
print('logL at truth         : {:.2f}'.format(log_likelihood(TRUTH)))
print('logL at box corner    : {:.2f}'.format(log_likelihood(PRIOR_LO)))
print()
print('{:<18} {:>10} {:>10} {:>10}'.format('parameter', 'prior lo', 'truth', 'prior hi'))
print('-' * 52)
for i, label in enumerate(LABELS):
    print('{:<18} {:>10.4g} {:>10.4g} {:>10.4g}'.format(label, PRIOR_LO[i], TRUTH[i], PRIOR_HI[i]))


quiet flux            : 15.372176
data vector           : 1 + 20 = 21 numbers, 4 parameters
one likelihood call   : 0.0597 s
logL at truth         : -15.38
logL at box corner    : -90624.53

parameter            prior lo      truth   prior hi
----------------------------------------------------
inc [deg]                  70         75         80
v_eq [m/s]               3600       4000       4400
lat_centre [deg]           21         25         29
ff                        0.1       0.12       0.14


In [ ]:
# =============================================================================
# CELL 2 -- run dynesty on the small box and check the recovery
# =============================================================================

import dynesty
from dynesty import utils as dyfunc

N_LIVE = 100
SAMPLE_METHOD = 'rwalk'   # fine inside a box this narrow. A real run over a WIDE
                          # prior wants 'rslice': v_eq and inc enter the Doppler map
                          # only as v_eq*sin(inc) (simulator.py:500), a curved ridge
                          # that ellipsoidal rejection sampling tracks poorly.
DLOGZ = 0.1               # stopping criterion on the remaining evidence

sampler = dynesty.NestedSampler(log_likelihood, prior_transform, N_DIM,
                                nlive=N_LIVE,
                                sample=SAMPLE_METHOD,
                                rstate=np.random.default_rng(SEED + 1))

t0 = time.time()
sampler.run_nested(dlogz=DLOGZ, print_progress=True)
t_run = time.time() - t0

results = sampler.results
n_calls = int(np.sum(results.ncall))

# nested sampling returns WEIGHTED samples: the raw count is not the effective
# sample size, and a large gap means the histograms are noisier than they look
weights = np.exp(results.logwt - results.logz[-1])
weights = weights / weights.sum()
ess = 1.0 / np.sum(weights ** 2)
samples = results.samples

print()
print('wall time        : {:.1f} s'.format(t_run))
print('iterations       : {:,}'.format(results.niter))
print('likelihood calls : {:,}   ({:.4f} s each)'.format(n_calls, t_run / n_calls))
print('log Z            : {:.3f} +/- {:.3f}'.format(results.logz[-1], results.logzerr[-1]))
print('effective samples: {:.0f} of {:,} weighted points'.format(ess, len(weights)))
print()

if ess < 50:
    print('*** ESS is small: the numbers below are noisy. Raise N_LIVE. ***')
    print()

print('{:<18} {:>12} {:>12} {:>12} {:>10}'.format(
    'parameter', 'truth', 'mean', 'std', 'bias/std'))
print('-' * 68)
for i, label in enumerate(LABELS):
    mean, cov = dyfunc.mean_and_cov(samples[:, i:i + 1], weights)
    posterior_mean = float(mean[0])
    posterior_std = float(np.sqrt(cov[0, 0]))
    bias = (posterior_mean - TRUTH[i]) / posterior_std if posterior_std > 0 else np.nan
    print('{:<18} {:>12.4g} {:>12.4g} {:>12.4g} {:>10.2f}'.format(
        label, TRUTH[i], posterior_mean, posterior_std, bias))

print()
print('A std close to the prior-box std means that parameter is UNCONSTRAINED and the')
print('test is handing the prior straight back. Box std, (hi-lo)/sqrt(12):')
print('  ', np.round((PRIOR_HI - PRIOR_LO) / np.sqrt(12.0), 4))

fig, axes = plt.subplots(1, N_DIM + 1, figsize=(4 * (N_DIM + 1), 3.2))
for i, label in enumerate(LABELS):
    axes[i].hist(samples[:, i], bins=30, weights=weights, density=True,
                 histtype='stepfilled', alpha=0.6)
    axes[i].axvline(TRUTH[i], color='r', lw=1.5)
    axes[i].set_xlim(PRIOR_LO[i], PRIOR_HI[i])
    axes[i].set_xlabel(label)
axes[0].set_ylabel('posterior density')
axes[-1].plot(results.logl, lw=1.0)
axes[-1].set_xlabel('iteration')
axes[-1].set_ylabel('logL of discarded point')
axes[-1].set_title('should rise monotonically')
fig.tight_layout()
plt.show()


 33%|███▎      | 732/2197 [11:48<36:18,  1.49s/it, bound: 4 | nc: 24 | ncall: 12318 | eff(%):  5.943 | loglstar: -48.080 | logz: -58.049 +/-  0.298 | dlogz: 36.357 >  0.100]     